# Day 40 — **pgvector with asyncpg**

**Phase 2 · Module 6: Agent Memory, Tools & MCP** · ~30 min

FAISS (Days 39–40) is great in-process but has no persistence, no HA, and can't filter vector search by SQL predicates. If your data already lives in Postgres, **pgvector** stores embeddings *next to* your relational rows and lets one query filter **and** rank by similarity. **asyncpg** is the fast async driver — the right client for the async agents from Day 35.

No Postgres in a notebook, so below is a small **asyncpg-shaped mock** (`create_pool`, `pool.acquire()`, `conn.execute/fetch`, `$1` params) that runs the real SQL patterns you'd send — `<=>` cosine distance, `ORDER BY ... LIMIT k`. Swap `MockPg` → `asyncpg` and the SQL is unchanged. Install for real: `pip install asyncpg pgvector` + a Postgres with the `vector` extension.

Today:
1. **Connection pool** — one pool for the agent's lifetime.
2. **Schema** — a `vector(N)` column alongside your data.
3. **Insert** embeddings with `$1` bound params.
4. **Nearest-neighbour query** — `ORDER BY embedding <=> $1 LIMIT 5`, with a SQL filter.

## 1. The asyncpg-shaped mock

Just enough of asyncpg to run real query text: a pool you `acquire()` as an async context manager, `execute` for writes, `fetch` for reads with `$1` params. It interprets the `<=>` distance query in Python so the notebook runs offline.

In [1]:
import asyncio, nest_asyncio, re, numpy as np
nest_asyncio.apply()

class Record(dict):
    """asyncpg returns Record; row['col'] access. dict is close enough."""

class _Conn:
    def __init__(self, store): self._s = store
    async def execute(self, sql, *args):
        s = sql.strip().lower()
        if s.startswith("create") or "extension" in s:
            return "OK"                                  # DDL: no-op in the mock
        if s.startswith("insert"):
            self._s["rows"].append({"id": args[0], "content": args[1],
                                    "customer_id": args[2], "embedding": np.asarray(args[3], "float32")})
            return "INSERT 0 1"
        return "OK"
    async def fetch(self, sql, *args):
        # emulate: SELECT ... [WHERE customer_id=$X] ORDER BY embedding <=> $Q LIMIT k
        rows = self._s["rows"]
        m = re.search(r"customer_id\s*=\s*\$(\d+)", sql)
        if m:
            cid = args[int(m.group(1)) - 1]
            rows = [r for r in rows if r["customer_id"] == cid]     # SQL filter
        q = np.asarray(args[0], "float32")               # $1 = query vector here
        def cos_dist(v):                                  # pgvector <=> operator
            denom = (np.linalg.norm(v) * np.linalg.norm(q)) or 1e-9
            return 1 - float(v @ q) / denom
        ranked = sorted(rows, key=lambda r: cos_dist(r["embedding"]))
        k = int(re.search(r"limit\s+(\d+)", sql.lower()).group(1))
        return [Record(id=r["id"], content=r["content"],
                       distance=round(cos_dist(r["embedding"]), 3)) for r in ranked[:k]]

class _Acquire:
    def __init__(self, store): self._s = store
    async def __aenter__(self): return _Conn(self._s)
    async def __aexit__(self, *e): return False

class MockPool:
    def __init__(self): self._store = {"rows": []}
    def acquire(self): return _Acquire(self._store)      # async with pool.acquire()
    async def close(self): pass

async def create_pool(dsn=None, **kw):                    # mirrors asyncpg.create_pool
    return MockPool()

print("mock ready — same surface as asyncpg.create_pool / pool.acquire / conn.fetch")

mock ready — same surface as asyncpg.create_pool / pool.acquire / conn.fetch


☝ The surface is exactly asyncpg's: `pool = await create_pool(dsn)`, `async with pool.acquire() as conn`, `await conn.fetch(sql, *args)` with `$1`-style params. Real asyncpg sends that SQL to Postgres; the mock interprets `<=>` (pgvector's **cosine distance** operator) in numpy. The only real-vs-mock difference is the constructor line — which is the whole point of testing against a faithful stand-in.

## 2. Schema — a vector column beside your data

pgvector adds a `vector(N)` type. Put it in the same table as your business columns so one query can filter on `customer_id` *and* rank by embedding. `N` must match your embedder's output dim (1536 for OpenAI, 384 for MiniLM).

In [2]:
DIM = 8   # tiny for the demo; real: 1536 (OpenAI) or 384 (MiniLM)

async def setup(pool):
    async with pool.acquire() as conn:
        await conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
        await conn.execute(f"""
            CREATE TABLE IF NOT EXISTS agent_memory (
                id          bigserial PRIMARY KEY,
                content     text NOT NULL,
                customer_id text NOT NULL,
                embedding   vector({DIM})
            )""")
        # real deployments add an ANN index for speed at scale:
        await conn.execute("CREATE INDEX IF NOT EXISTS idx_mem ON agent_memory "
                           "USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100)")
    return "schema ready"

pool = asyncio.run(create_pool("postgresql://localhost/bank"))
print(asyncio.run(setup(pool)))

schema ready


☝ Two things to carry to production. The `vector(DIM)` dim is **fixed at table creation** and must equal your embedder's output — mismatch is a hard error on insert, so pick the model first. And the `ivfflat` (or `hnsw`) index is what makes similarity search fast at scale: without it Postgres does an exact scan of every row (fine for thousands, fatal for millions). `vector_cosine_ops` tells the index to optimise for the `<=>` cosine operator you'll query with — match the index opclass to the distance operator you use.

## 3. Insert embeddings with bound params

Always use `$1, $2, ...` bound parameters — never string-format values into SQL (that's the SQL-injection door from Day 27). asyncpg sends params separately from the query text.

In [3]:
def fake_embed(text: str, dim=DIM) -> list[float]:
    """Stand-in embedder; real: sentence-transformers / Bedrock Titan embeddings."""
    import hashlib
    v = np.zeros(dim, "float32")
    for tok in text.lower().split():
        v[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    n = np.linalg.norm(v) or 1e-9
    return (v / n).tolist()

INSIGHTS = [
    (1, "SME manufacturer arrears recover within two quarters", "C-4417"),
    (2, "overdraft fee dispute refunded after review", "C-4417"),
    (3, "sanctions review flagged shell company", "C-9001"),
    (4, "mortgage application needs two years accounts", "C-8820"),
]

async def insert_all(pool):
    async with pool.acquire() as conn:
        for id_, content, cust in INSIGHTS:
            await conn.execute(
                "INSERT INTO agent_memory (id, content, customer_id, embedding) "
                "VALUES ($1, $2, $3, $4)",
                id_, content, cust, fake_embed(content))       # $4 = the vector
    return "inserted"

print(asyncio.run(insert_all(pool)))

inserted


☝ The `$1..$4` placeholders are non-negotiable: parameters are sent to Postgres separately from the SQL text, so a malicious `content` value can't alter the query — the injection defence from Day 27, enforced by the driver. Note the embedding goes in as a plain list of floats (`$4`); the `pgvector` Python adapter (or asyncpg codec) converts it to the `vector` type on the wire. Batch inserts in production use `conn.executemany` or `COPY` for throughput.

## 4. Nearest-neighbour query — the payoff

`ORDER BY embedding <=> $1 LIMIT k` returns the k closest rows by cosine distance. Add a `WHERE customer_id = $2` and you get **filtered** vector search in one query — the thing FAISS alone can't do.

In [4]:
async def search(pool, query: str, k=3, customer_id: str | None = None):
    qvec = fake_embed(query)
    async with pool.acquire() as conn:
        if customer_id:
            rows = await conn.fetch(
                "SELECT id, content, embedding <=> $1 AS distance FROM agent_memory "
                "WHERE customer_id = $2 ORDER BY embedding <=> $1 LIMIT " + str(k),
                qvec, customer_id)
        else:
            rows = await conn.fetch(
                "SELECT id, content, embedding <=> $1 AS distance FROM agent_memory "
                "ORDER BY embedding <=> $1 LIMIT " + str(k),
                qvec)
    return [(r["distance"], r["content"]) for r in rows]

print("global semantic search 'arrears recovery':")
for dist, content in asyncio.run(search(pool, "arrears recovery outlook")):
    print(f"  d={dist:.3f}  {content}")

print("\nscoped search for C-4417 only:")
for dist, content in asyncio.run(search(pool, "fee complaint", customer_id="C-4417")):
    print(f"  d={dist:.3f}  {content}")

global semantic search 'arrears recovery':
  d=0.388  mortgage application needs two years accounts
  d=0.478  SME manufacturer arrears recover within two quarters
  d=0.564  sanctions review flagged shell company

scoped search for C-4417 only:
  d=0.360  SME manufacturer arrears recover within two quarters
  d=0.553  overdraft fee dispute refunded after review


☝ Lower `distance` = closer (it's a *distance*, not a similarity — `<=>` returns `1 - cosine`). The `WHERE customer_id = $2` clause is exactly what makes pgvector worth choosing over FAISS: **filtered ANN search in one round-trip** — episodic-style scoping (Day 39) and semantic ranking (Day 40) in a single SQL statement, transactionally consistent with the rest of your Postgres data. That combination — vectors + relational filters + one datastore — is the pgvector pitch for agent memory.

## Recap

| Task | pgvector + asyncpg |
|---|---|
| Reuse connections | `await create_pool(dsn)`, `async with pool.acquire()` |
| Store vectors by data | `vector(N)` column in your table |
| Fast at scale | `ivfflat`/`hnsw` index, opclass = distance op |
| Safe inserts | `$1..$N` bound params (never f-string SQL) |
| Filtered similarity | `WHERE ... ORDER BY embedding <=> $1 LIMIT k` |

**Swap to prod:** `MockPg` → `import asyncpg`; the SQL is identical. **FAISS vs pgvector:** FAISS for embedded/in-process; pgvector when you're already on Postgres and need SQL filters + durability. Tomorrow (Day 41): tool schema design with Pydantic — precise, validated tool inputs for agents.